# SILK 스타일 손 관절 Motion In-betweening

이 노트북은 **Akhoundi et al., "SILK: Smooth InterpoLation frameworK for motion in-betweening"
(CVPR 2025 HuMoGen Workshop, arXiv:2506.09075)** 논문에 명시된 아키텍처/학습 스펙을
최대한 그대로 따르되, 손 관절(회전 정보 없는 3D 좌표만 있는) 도메인에 맞게 조정한 구현입니다.

**공식 코드는 공개되어 있지 않습니다** (GitHub 저장소, 프로젝트 페이지 모두 확인함).
아래는 논문 Section 3, 4.2에 명시된 스펙을 그대로 반영한 재구현입니다.

## 논문 대비 우리가 그대로 따른 것
- Transformer 인코더 **6층, 8-head**, `d_model=1024`, `d_ff=4096`, **Pre-LN**
- **AdamW + Noam 학습률 스케줄러**, batch size **64**
- 입력 구조: **C개 컨텍스트 프레임 + M개 빈(0) 프레임 + 목표 키프레임 1개**
- 빈 프레임은 **0으로 채우고 attention masking 없음** (모든 프레임이 서로 attend)
- **단일 L1 손실**
- 학습 시 M(가림 길이)을 5~30 사이에서 균일 샘플링, 평가는 5/15/30/45 고정
- 데이터 슬라이스 오프셋 5 (논문 대비 4배 촘촘한 샘플링)

## 손 도메인에 맞게 조정한 것 (논문과 다른 지점, 명시)
- **회전(quaternion/6D) 특징 전부 제외** — 우리 데이터(SHREC, H2O, InterHand2.6M 등)는
  3D 관절 좌표만 제공하고 회전 정보가 없음. 원 논문의 `d_in=18J+8`, `d_out=9J+4` 대신
  **위치+속도만 사용**: `d_in = 6J`(위치 3J + 선속도 3J), `d_out = 3J`(위치만).
- **"루트를 지면에 투영"하는 개념 없음** — 몸은 골반을 지면에 투영해 root를 만들지만,
  손에는 이런 개념이 없어 **손목을 root 삼아 상대좌표로 정규화**(기존 파이프라인과 동일한 방식).
- **Relative positional encoding의 정확한 구현 방식은 논문에 상세 공개 안 됨** — 학습 가능한
  절대 위치 임베딩(learned absolute positional embedding)으로 근사 구현. (논문은 [25]의 방식을
  따른다고만 언급하고 구체적 수식은 미공개)


## 1. 환경 설정

In [ ]:

!pip install -q gdown

import os, math, pickle, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


Mounted at /content/drive
device: cuda


## 2. 설정값

`N_JOINTS`는 사용하는 데이터셋에 맞춰 바꿔주세요 (SHREC=22, H2O/MediaPipe/MANO 계열=21).


In [ ]:

N_JOINTS = 21          # 데이터셋에 맞게 조정 (SHREC=22, H2O/InterHand=21)
COORD_DIM = 3
J = N_JOINTS

D_IN = 6 * J            # 위치(3J) + 선속도(3J)  -- 논문의 18J+8(회전 포함)을 위치/속도만으로 축소
D_OUT = 3 * J           # 위치만 (논문처럼 output에는 속도 미포함)

D_MODEL = 1024          # 논문 Section 4.2
N_HEADS = 8              # 논문 Section 4.2
N_LAYERS = 6              # 논문 Section 4.2
D_FF = 4096              # 논문 Section 4.2
DROPOUT = 0.1             # 논문에 명시 안 됨 -> Transformer 관례값 사용
BATCH_SIZE = 64          # 논문 Section 4.2

C_CONTEXT = 10            # 논문 Section 4.3: 컨텍스트 프레임 10개
M_TRAIN_RANGE = (5, 30)    # 논문 Section 4.3: 학습 시 가림 길이 5~30 균일 샘플링
M_EVAL_LENGTHS = [5, 10, 20, 30]  # 우리 프로젝트에서 다뤄온 L값 기준으로 평가

MAX_SEQ_LEN = C_CONTEXT + max(M_TRAIN_RANGE[1], max(M_EVAL_LENGTHS)) + 1  # +1: 목표 키프레임
print("모델이 다루는 최대 시퀀스 길이:", MAX_SEQ_LEN)


모델이 다루는 최대 시퀀스 길이: 41


## 3. 위치 인코딩

논문은 "learned relative positional encoding"(Qin et al. 2022 방식)을 쓴다고 서술하지만
정확한 구현 수식은 공개하지 않았습니다.

**중요한 구현 디테일(직접 겪은 버그로 확인됨)**: 절대 위치(0,1,2,...)를 그대로 쓰면,
같은 인덱스가 M(가림 길이)에 따라 "목표 키프레임"이 되기도 하고 "가려진 예측 대상"이
되기도 해서 학습 신호가 서로 충돌합니다. 예를 들어 인덱스 15는 M=5일 때는 목표
키프레임(정답 그대로)이지만, M=6~30일 때는(전체 학습의 96%) gap 프레임(0으로 채워짐)
이라, 모델이 "이 위치는 대체로 예측 대상"이라고 학습해버려 M이 작은 조건(L5 등)의
성능이 오히려 나빠지는 문제가 실측으로 확인되었습니다.

**해결**: 절대 위치가 아니라 **목표 키프레임까지 남은 상대 거리**를 위치 정보로 사용합니다.
목표 프레임은 항상 상대위치 0, 그 직전 프레임은 항상 -1, 이런 식으로 M과 무관하게
"목표까지 N프레임 남았다"는 의미가 항상 일관되게 유지됩니다.


In [ ]:

class RelativePositionalEncoding(nn.Module):
    '''목표 키프레임을 기준(상대위치 0)으로 한 학습 가능한 위치 임베딩.
    컨텍스트/gap 프레임은 목표 대비 음수 상대위치를 가짐.'''
    def __init__(self, d_model, max_len=200):
        super().__init__()
        self.max_len = max_len
        self.pos_embedding = nn.Embedding(2 * max_len + 1, d_model)

    def forward(self, x, rel_pos):
        # x: (B, T, d_model), rel_pos: (B, T) long, 목표 프레임 기준 상대 위치(<=0)
        idx = (rel_pos + self.max_len).clamp(0, 2 * self.max_len)
        return x + self.pos_embedding(idx)


## 4. SILK 모델 본체

논문 Figure 2, Section 3.2 그대로:
- 프레임 시퀀스를 `d_model`로 linear projection
- 위치 임베딩 추가
- **단일** Transformer 인코더 (마스킹 없음 — 0으로 채워진 프레임도 다른 모든 프레임과 attend)
- linear projection으로 `d_out` 복원


In [ ]:

class SILKHand(nn.Module):
    def __init__(self, d_in=D_IN, d_out=D_OUT, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
                 dropout=DROPOUT, max_len=MAX_SEQ_LEN):
        super().__init__()
        self.input_proj = nn.Linear(d_in, d_model)
        self.pos_enc = RelativePositionalEncoding(d_model, max_len=max_len)

        # 논문: "layer normalization takes place prior to attention and
        # feedforward operations" -> Pre-LN, norm_first=True
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,   # Pre-LN
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, d_out)

    def forward(self, x, rel_pos, valid_mask=None):
        # x: (B, T, d_in) -- gap 구간은 이미 0으로 채워진 상태
        # rel_pos: (B, T) -- 목표 키프레임 기준 상대 위치 (0=목표, 음수=그 이전)
        # valid_mask: (B, T) bool, True=실제 데이터, False=배치 패딩
        # 논문: "we are not masking the in-between frames" -> gap 프레임에 대한 마스킹은 없음.
        # 다만 배치 내 길이를 맞추기 위한 패딩 프레임은 attention에서 제외하는 것이 맞음.
        h = self.input_proj(x)
        h = self.pos_enc(h, rel_pos)
        key_padding_mask = ~valid_mask if valid_mask is not None else None
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        out = self.output_proj(h)
        return out


## 5. Noam 학습률 스케줄러

논문: "AdamW optimizer with a noam learning rate scheduler same as [25]"
(Qin et al. 2022가 쓴 것과 동일 -- 원조는 Vaswani et al. "Attention is All You Need"의 스케줄).


In [ ]:

class NoamScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=4000, factor=1.0):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.factor * (self.d_model ** -0.5) * min(
            self.step_num ** -0.5,
            self.step_num * (self.warmup_steps ** -1.5),
        )
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        return lr


## 6. 데이터 파이프라인

논문 구조 그대로: 원본 시퀀스에서 **C개 컨텍스트 프레임 + M개 프레임(정답, 학습 목표) + 목표
키프레임 1개**를 뽑습니다. 입력으로 모델에 들어가는 것은 `[컨텍스트(C) | 0으로 채운 gap(M) |
목표 키프레임(1)]`이고, 정답은 `[컨텍스트(C) | 실제 gap 값(M) | 목표 키프레임(1)]`입니다.

`sequences`는 `(frames, N_JOINTS, 3)` numpy 배열들의 리스트라고 가정합니다 (SHREC/H2O/InterHand
로더에서 이미 만들어둔 것을 그대로 재사용하면 됩니다).


In [ ]:

def normalize_sequence(seq, wrist_idx=0):
    '''손목(wrist_idx) 기준 상대좌표 + 평균 관절 거리로 스케일 정규화.
    특정 손가락 인덱스에 의존하지 않는 안전한 스케일 기준을 사용.'''
    wrist = seq[:, wrist_idx:wrist_idx+1, :]
    relative = seq - wrist
    dists = np.linalg.norm(relative, axis=-1)          # (frames, J)
    scale = dists.mean(axis=-1, keepdims=True)          # (frames, 1)
    scale = np.clip(scale, 1e-6, None)
    return relative / scale[:, :, None]


def build_silk_sample(seq, C, M, offset_start, rng):
    '''seq: (frames, J, 3) 정규화된 시퀀스.
    offset_start: 컨텍스트 시작 위치.
    반환: input_features (T, 6J), target_positions (T, 3J), gap_mask (T,) bool
          T = C + M + 1
    '''
    total_len = C + M + 1
    if offset_start + total_len > seq.shape[0]:
        return None

    window = seq[offset_start: offset_start + total_len]   # (T, J, 3)
    T = window.shape[0]

    # 정답 위치
    target_positions = window.reshape(T, -1).astype(np.float32)   # (T, 3J)

    # 입력용 위치: gap(M) 구간만 0으로
    input_positions = target_positions.copy()
    gap_mask = np.zeros(T, dtype=bool)
    gap_mask[C: C + M] = True
    input_positions[gap_mask] = 0.0

    # 속도 특징: 연속 프레임 차분 (첫 프레임 속도는 0)
    # 논문 4.5.2 "Input Vel" 설정을 따름: 속도는 입력에만 포함, 출력엔 미포함
    velocity = np.zeros_like(input_positions)
    velocity[1:] = input_positions[1:] - input_positions[:-1]
    # gap 프레임의 속도도 계산상 이상값이 섞이므로 0으로 통일 (정보 누수 방지)
    velocity[gap_mask] = 0.0

    input_features = np.concatenate([input_positions, velocity], axis=-1)  # (T, 6J)

    # 목표 키프레임(인덱스 C+M) 기준 상대 위치: 목표=0, 그 이전은 음수
    target_idx = C + M
    rel_pos = np.arange(T, dtype=np.int64) - target_idx

    return input_features, target_positions, gap_mask, rel_pos


class SILKHandDataset(Dataset):
    '''
    sequences: list of (frames, J, 3) ndarray
    mode='train': 매 __getitem__마다 M을 M_TRAIN_RANGE에서 랜덤 샘플링, 시작 위치도 랜덤
    mode='eval': 고정된 M(eval_length)과 겹치지 않는 여러 위치로 고정 샘플 생성
    '''
    def __init__(self, sequences, C=C_CONTEXT, mode="train",
                 m_range=M_TRAIN_RANGE, eval_length=None, samples_per_epoch=4000, seed=SEED):
        self.sequences = [normalize_sequence(s) for s in sequences if s.shape[0] >= C + 6]
        self.C = C
        self.mode = mode
        self.m_range = m_range
        self.eval_length = eval_length
        self.rng = np.random.default_rng(seed)

        if mode == "train":
            self.samples_per_epoch = samples_per_epoch
        else:
            # eval: 겹치지 않게 타일링해서 고정 샘플 목록을 미리 만들어둠
            assert eval_length is not None
            self.fixed_samples = []
            total_len = C + eval_length + 1
            for seq in self.sequences:
                n_windows = seq.shape[0] // total_len
                for i in range(n_windows):
                    start = i * total_len
                    sample = build_silk_sample(seq, C, eval_length, start, self.rng)
                    if sample is not None:
                        self.fixed_samples.append(sample)

    def __len__(self):
        if self.mode == "train":
            return self.samples_per_epoch
        return len(self.fixed_samples)

    def __getitem__(self, idx):
        if self.mode == "train":
            # 매번 랜덤하게 시퀀스, M, 시작 위치를 뽑음
            for _ in range(20):  # 최대 20번 재시도 (너무 짧은 시퀀스 방어)
                seq = self.sequences[self.rng.integers(0, len(self.sequences))]
                M = int(self.rng.integers(self.m_range[0], self.m_range[1] + 1))
                total_len = self.C + M + 1
                if seq.shape[0] < total_len:
                    continue
                start = int(self.rng.integers(0, seq.shape[0] - total_len + 1))
                sample = build_silk_sample(seq, self.C, M, start, self.rng)
                if sample is not None:
                    break
            else:
                raise RuntimeError("적합한 시퀀스를 찾지 못했습니다 (시퀀스가 너무 짧음)")
        else:
            sample = self.fixed_samples[idx]

        input_features, target_positions, gap_mask, rel_pos = sample
        return (
            torch.from_numpy(input_features),
            torch.from_numpy(target_positions),
            torch.from_numpy(gap_mask),
            torch.from_numpy(rel_pos),
        )


def collate_pad(batch):
    '''배치 안에서 시퀀스 길이(M이 달라서)가 다를 수 있으므로 0-패딩.
    rel_pos의 패딩 값은 절대 실제 목표(0)나 유효 구간과 안 겹치게 아주 큰 음수로 채움
    (valid_mask로 어차피 손실 계산에서 제외되지만, 안전하게 티 나는 값을 사용).'''
    inputs, targets, masks, rel_positions = zip(*batch)
    lengths = [x.shape[0] for x in inputs]
    max_len = max(lengths)

    pad_inputs, pad_targets, pad_masks, pad_valid, pad_relpos = [], [], [], [], []
    for inp, tgt, m, rp in zip(inputs, targets, masks, rel_positions):
        pad_len = max_len - inp.shape[0]
        pad_inputs.append(torch.nn.functional.pad(inp, (0, 0, 0, pad_len)))
        pad_targets.append(torch.nn.functional.pad(tgt, (0, 0, 0, pad_len)))
        pad_masks.append(torch.nn.functional.pad(m, (0, pad_len), value=False))
        pad_relpos.append(torch.nn.functional.pad(rp, (0, pad_len), value=-9999))
        valid = torch.zeros(max_len, dtype=torch.bool)
        valid[:inp.shape[0]] = True
        pad_valid.append(valid)

    return (torch.stack(pad_inputs), torch.stack(pad_targets),
            torch.stack(pad_masks), torch.stack(pad_valid), torch.stack(pad_relpos))


## 7. 손실 함수 & 평가지표

논문: "단일 L1 손실을 모든 특징에 대해 적용" — 컨텍스트/목표 프레임을 포함한 **전체 시퀀스**에
대해 L1을 계산합니다 (마스킹은 attention에서만 없을 뿐, loss 자체는 전체 프레임 기준이라는
논문 서술을 그대로 따름). 패딩된 부분은 `valid_mask`로 제외합니다.

MPJPE는 우리 프로젝트의 기존 물리 기반 베이스라인(칼만 필터 등)과 비교 가능하도록 **gap
구간만** 별도로 계산합니다.


In [ ]:

def l1_loss_full(pred, target, valid_mask):
    diff = torch.abs(pred - target)                # (B, T, 3J)
    diff = diff.mean(dim=-1)                        # (B, T)
    diff = diff * valid_mask.float()
    return diff.sum() / valid_mask.float().sum().clamp(min=1.0)


def masked_mpjpe(pred, target, gap_mask, n_joints=N_JOINTS):
    B, T, _ = pred.shape
    pred = pred.reshape(B, T, n_joints, 3)
    target = target.reshape(B, T, n_joints, 3)
    per_joint_err = torch.norm(pred - target, dim=-1)   # (B, T, J)
    per_frame_err = per_joint_err.mean(dim=-1)           # (B, T)
    masked = per_frame_err * gap_mask.float()
    return (masked.sum() / gap_mask.float().sum().clamp(min=1.0)).item()


## 8. 학습 루프

In [ ]:

def train_silk(train_sequences, val_sequences, epochs=30,
               samples_per_epoch=4000, warmup_steps=4000, lr_factor=1.0):

    train_ds = SILKHandDataset(train_sequences, mode="train",
                                samples_per_epoch=samples_per_epoch)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=collate_pad, drop_last=True)

    val_loaders = {
        L: DataLoader(
            SILKHandDataset(val_sequences, mode="eval", eval_length=L),
            batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pad,
        )
        for L in M_EVAL_LENGTHS
    }

    model = SILKHand().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.0)  # lr은 스케줄러가 매 step 덮어씀
    scheduler = NoamScheduler(optimizer, d_model=D_MODEL, warmup_steps=warmup_steps, factor=lr_factor)

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0

        for inputs, targets, gap_masks, valid_masks, rel_positions in train_loader:
            inputs = inputs.to(DEVICE).float()
            targets = targets.to(DEVICE).float()
            valid_masks = valid_masks.to(DEVICE)
            rel_positions = rel_positions.to(DEVICE)

            optimizer.zero_grad()
            pred = model(inputs, rel_positions, valid_masks)
            loss = l1_loss_full(pred, targets, valid_masks)
            loss.backward()
            scheduler.step()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        model.eval()
        val_report = {}
        with torch.no_grad():
            for L, loader in val_loaders.items():
                total_err, n = 0.0, 0
                for inputs, targets, gap_masks, valid_masks, rel_positions in loader:
                    inputs = inputs.to(DEVICE).float()
                    targets = targets.to(DEVICE).float()
                    gap_masks = gap_masks.to(DEVICE)
                    valid_masks = valid_masks.to(DEVICE)
                    rel_positions = rel_positions.to(DEVICE)
                    pred = model(inputs, rel_positions, valid_masks)
                    total_err += masked_mpjpe(pred, targets, gap_masks)
                    n += 1
                val_report[L] = total_err / max(n, 1)

        print(f"[Epoch {epoch+1}/{epochs}] train_L1={epoch_loss/max(n_batches,1):.4f} | "
              + " ".join(f"val_MPJPE_L{L}={v:.4f}" for L, v in val_report.items()))

    return model


## 9. 실행

`train_sequences`, `val_sequences`는 `(frames, N_JOINTS, 3)` ndarray들의 리스트여야 합니다.
기존에 만들어둔 SHREC / H2O 로더의 출력(정규화 이전의 raw 시퀀스)을 그대로 넣으면 됩니다.


In [ ]:
import pickle

with open("/content/drive/MyDrive/KUBIG Summer Contest/raw_sequences_H2O_train.pkl", "rb") as f:
    train_sequences = pickle.load(f)
with open("/content/drive/MyDrive/KUBIG Summer Contest/raw_sequences_H2O_val.pkl", "rb") as f:
    val_sequences = pickle.load(f)

model = train_silk(train_sequences, val_sequences, epochs=30)
torch.save(model.state_dict(), "/content/drive/MyDrive/KUBIG Summer Contest/silk_hand_pretrained_h2o_new.pt")

/tmp/ipykernel_435/3850145133.py:20: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


[Epoch 1/30] train_L1=0.6368 | val_MPJPE_L5=0.5096 val_MPJPE_L10=0.4311 val_MPJPE_L20=0.4598 val_MPJPE_L30=0.5584
[Epoch 2/30] train_L1=0.2294 | val_MPJPE_L5=0.3112 val_MPJPE_L10=0.2310 val_MPJPE_L20=0.2384 val_MPJPE_L30=0.2839
[Epoch 3/30] train_L1=0.1731 | val_MPJPE_L5=0.2119 val_MPJPE_L10=0.1657 val_MPJPE_L20=0.1864 val_MPJPE_L30=0.2175
[Epoch 4/30] train_L1=0.1490 | val_MPJPE_L5=0.1641 val_MPJPE_L10=0.1397 val_MPJPE_L20=0.1571 val_MPJPE_L30=0.1889
[Epoch 5/30] train_L1=0.1318 | val_MPJPE_L5=0.1446 val_MPJPE_L10=0.1227 val_MPJPE_L20=0.1432 val_MPJPE_L30=0.1714
[Epoch 6/30] train_L1=0.1190 | val_MPJPE_L5=0.1262 val_MPJPE_L10=0.1112 val_MPJPE_L20=0.1275 val_MPJPE_L30=0.1533
[Epoch 7/30] train_L1=0.1067 | val_MPJPE_L5=0.1175 val_MPJPE_L10=0.1078 val_MPJPE_L20=0.1238 val_MPJPE_L30=0.1450
[Epoch 8/30] train_L1=0.0970 | val_MPJPE_L5=0.0990 val_MPJPE_L10=0.0973 val_MPJPE_L20=0.1103 val_MPJPE_L30=0.1374
[Epoch 9/30] train_L1=0.0880 | val_MPJPE_L5=0.0983 val_MPJPE_L10=0.0882 val_MPJPE_L20=0.

In [ ]:
from google.colab import runtime
runtime.unassign()